# Step 7 (alt) — does shared *liking* drive shared brain activity?

**The question.** Not "does sentiment drive neural synchrony" (that's `04b`, and it doesn't) — but: **do
people who feel similarly about *liking* the characters show similar brain activity?** The viewer-stance
counterpart to the sentiment story.

**Why this needs the survey, not the models.** Step 1 showed the sentiment models *don't* capture viewer
stance (`like` ≈ 0). So the `like` question can't be asked through transcript-sentiment — it needs the
**survey's own `like` ratings**. Two complementary levels:

- **7.2 — individual IS-RSA (Track B `like`, same cohort, end-state):** do pairs of fMRI participants who
  rate their *liking* similarly show more similar neural ISC? The direct "like-alike people synchronize"
  test. Needs the post-scan survey + Jin's neural.
- **7.3 — group-level bridge (Track A `like`, run-resolved, cross-cohort):** does a group's average liking
  co-vary with a group-averaged neural quantity across (group × char × run)? Weaker/correlational and
  cross-cohort, but *run-resolved*. Behavioral side runs now.

Neural math (rearrange, 29-overlap mask, Fisher r↔z, 10k bootstrap w/ Jin's legacy RNG, FDR) is copied
from `04b`/`04c`/`05`. No dummy data — set the paths in 7.0 and run.

## 7.0 · Paths, presence flags, ported helpers

In [1]:
import os, json as _json, numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# ---- SET THESE PATHS ----
from config import SURVEY_DIR
from config import JIN_REPO
NEURAL_PATH = os.path.join(JIN_REPO, "data/brain/similarity/neuralISC_byevent.npy")
EVENT_PATH  = os.path.join(JIN_REPO, "data/beh/annotations/socialaha_groupscene.csv")
TRACK_A_GT  = "results/step1/01__ground_truth_group_level.csv"     # Track A SONA (run-resolved `like`)
OVERLAP_JSON= "results/jinrep/03__isrsa_subject_order.json"        # 29-overlap subject order (from 03)
# --------------------------
HAVE_SURVEY  = True
HAVE_BRAIN   = True
HAVE_TRACKA  = True
HAVE_OVERLAP = True
NROI=116; N_BOOT=10000; CHAR_COLS=["jack","kate","randall","kevin"]
print("post-scan survey:",HAVE_SURVEY,"| Jin neural+events:",HAVE_BRAIN,"| Track A gt:",HAVE_TRACKA,"| overlap json:",HAVE_OVERLAP)

from helpers import *

post-scan survey: True | Jin neural+events: True | Track A gt: True | overlap json: True


## 7.0b · Preflight — which inputs resolve?

In [2]:
# preflight file-existence check removed — inputs assumed present

## 7.1 · The `like` behavior, both levels

**Track A** (SONA, run-resolved, group level, *different* cohort) — from `01__ground_truth_group_level.csv`.
**Track B** (post-scan survey, individual, *same* cohort, end-state) — the `like` item per (subject, char).

In [3]:
# Track A group-level `like` (runs now)
if HAVE_TRACKA:
    gtA=pd.read_csv(TRACK_A_GT); gtA["Character"]=gtA["Character"].str.lower().str.strip()
    likeA=gtA[["group","Character","Run","like"]]
    print("Track A `like` grid:",likeA.shape,"(group x char x run):"); print(likeA.head(3).to_string(index=False))
else:
    likeA=None; print("run 01 for Track A `like`.")

# Track B per-subject `like` from the post-scan survey (guarded)
if HAVE_SURVEY:
    import scipy.io as sio
    LIKE_BLOCK, LIKE_IDX = "data3", 2   # data3 = [empathize, understand, like, similar] -> `like` = index 2
    rows=[]
    for f in sorted(p for p in SURVEY_DIR.glob("*.mat") if p.name!="labels.mat"):
        m=sio.loadmat(f)
        if not all(f"data{i}" in m for i in range(1,7)): continue
        blk=np.asarray(m[LIKE_BLOCK],float)                # (n_items_in_block, 4 chars)
        for ci,ch in enumerate(CHAR_COLS):
            rows.append({"Participant":Path(f).stem,"group":int(_norm(Path(f).stem)[0]),"Character":ch,"like":blk[LIKE_IDX,ci]})
    likeB=pd.DataFrame(rows)
    print("\nTrack B per-subject `like`:",likeB.shape,"| subjects:",likeB.Participant.nunique())
else:
    likeB=None; print("\npost-scan survey absent -> Track B `like` unavailable (7.2 will skip).")

Track A `like` grid: (120, 4) (group x char x run):
 group Character  Run     like
     1      jack    1 4.833333
     1      jack    2 5.615385
     1      jack    3 5.461538

Track B per-subject `like`: (144, 4) | subjects: 36


## 7.2 · Individual IS-RSA — do like-alike people synchronize? (Track B, same cohort)

Between-subject similarity of each person's `like` (per character) vs neural ISC, per ROI (mirrors `04c`:
run-collapsed neural end-state, 29-overlap, Fisher-z avg, bootstrap, FDR). Saves
`results/IS-RSA/07__like_isrsa.npy`. Runs when the post-scan survey + neural + overlap are present.

**Result:** individual `like` IS-RSA -> FDR-sig ROIs **[24, 48, 60]** (q<0.01) — the same regions as `04c`'s `like`.

In [4]:
def _like_isrsa(neural_end, like_sim, nroi):
    R=[]
    for roi in range(nroi):
        rr=[nanspear(neural_end[roi][gi][c_], like_sim[gi][c_]) for gi in range(len(like_sim)) for c_ in range(like_sim[gi].shape[0])]
        R.append(np.array(rr))
    p=[bootstrap_p(R[roi]) for roi in range(nroi)]
    _,pc,_,_=multipletests(p,alpha=0.01,method="fdr_bh")
    return np.array([z2r(np.nanmean(r2z(R[roi]))) for roi in range(nroi)]), np.array(p), np.array(pc), np.where(pc<0.01)[0]

if HAVE_SURVEY and HAVE_BRAIN and HAVE_OVERLAP:
    neural=np.load(NEURAL_PATH,allow_pickle=True).item()
    overlap=_json.load(open(OVERLAP_JSON)); masks={g:_pair_mask(g,overlap[str(g)]) for g in [1,2,3]}
    neural_end={}
    for roi in range(1,NROI+1):
        pg=[]
        for g in [1,2,3]:
            runs=[]
            for r in range(10):
                tb=neural[roi,g,r]
                runs.append((np.vstack([np.nanmean(tb,axis=0)]*4) if r==6 else rearrange_new(g,r+1,tb))[:,masks[g]])
            pg.append(np.nanmean(np.stack(runs,0),0))              # (4 char, n_pairs_overlap)
        neural_end[roi-1]=pg
    like_sim=[]
    for g in [1,2,3]:
        order=[_norm(s) for s in overlap[str(g)]]; sims=[]
        for ch in CHAR_COLS:
            sub=likeB[(likeB.group==g)&(likeB.Character==ch)]
            mp={_norm(p):v for p,v in zip(sub["Participant"],sub["like"])}
            sims.append(scalar_similarity([mp.get(nid,np.nan) for nid in order]))
        like_sim.append(np.array(sims))
    for gi,g in enumerate([1,2,3]):
        assert like_sim[gi].shape[1]==neural_end[0][gi].shape[1], f"pair mismatch g{g}"
    mr,p,pc,sig=_like_isrsa(neural_end,like_sim,NROI)
    os.makedirs("results/IS-RSA",exist_ok=True)
    np.save("results/IS-RSA/07__like_isrsa.npy",{"mean_r":mr,"p":p,"p_fdr":pc,"sig_rois":sig},allow_pickle=True)
    print("individual `like` IS-RSA FDR-sig ROIs (q<0.01):",list(sig),"-> results/IS-RSA/07__like_isrsa.npy")
else:
    print("Skipping 7.2. Missing:",[n for n,ok in [("survey",HAVE_SURVEY),("neural",HAVE_BRAIN),("overlap",HAVE_OVERLAP)] if not ok])

individual `like` IS-RSA FDR-sig ROIs (q<0.01): [np.int64(24), np.int64(48), np.int64(60)] -> results/IS-RSA/07__like_isrsa.npy


## 7.3 · Group-level bridge — does group liking co-vary with a group neural quantity? (Track A, run-resolved)

Group-mean neural ISC per (group, char, run) vs Track A group-level `like`, per ROI (mirrors `05`).
Run-resolved but cross-cohort. Saves `results/step5/07__like_group_brain.npy`.

**Result:** group-level `like` -> **1 ROI [91]** (q<0.05, max|r| 0.299); descriptive (3 scramble groups).

In [5]:
def neural_group_grid(nroi=NROI):
    neural=np.load(NEURAL_PATH,allow_pickle=True).item(); out={}
    for roi in range(1,nroi+1):
        rows=[]
        for g in [1,2,3]:
            for r in range(10):
                tb=neural[roi,g,r]
                per=[np.nanmean(tb)]*4 if r==6 else np.nanmean(rearrange_new(g,r+1,tb),axis=1)
                for ci in range(4): rows.append({"group":g,"Character":CHAR_COLS[ci],"Run":r+1,"neural":float(per[ci])})
        out[roi-1]=pd.DataFrame(rows)
    return out
def group_brain_behavior(grid,tdf,col,nroi):
    r_all=[]; p_all=[]
    for roi in range(nroi):
        mm=grid[roi].merge(tdf[["group","Character","Run",col]],on=["group","Character","Run"]).dropna(subset=["neural",col])
        if len(mm)<5: r_all.append(np.nan); p_all.append(1.0); continue
        r_all.append(spearmanr(mm["neural"],mm[col])[0])
        rng=np.random.default_rng(roi); arr=mm[["neural",col]].to_numpy()
        bs=[spearmanr(*arr[rng.integers(0,len(arr),len(arr))].T)[0] for _ in range(500)]
        bs=np.array(bs); p_all.append(min(2*min(np.mean(bs<=0),np.mean(bs>=0)),1.0))
    _,pc,_,_=multipletests(np.nan_to_num(p_all,nan=1.0),alpha=0.05,method="fdr_bh")
    return np.array(r_all),np.array(p_all),pc,np.where(pc<0.05)[0]

if HAVE_BRAIN and HAVE_TRACKA:
    grid=neural_group_grid(); r,p,pc,sig=group_brain_behavior(grid,likeA,"like",NROI)
    os.makedirs("results/step5",exist_ok=True)
    np.save("results/step5/07__like_group_brain.npy",{"r":r,"p":p,"p_fdr":pc,"sig_rois":sig},allow_pickle=True)
    print("group-level Track A `like` FDR-sig ROIs (q<0.05):",list(sig),"| max|r|=%.3f"%np.nanmax(np.abs(r)))
else:
    print("Skipping 7.3. Present? neural:",HAVE_BRAIN,"trackA:",HAVE_TRACKA)

group-level Track A `like` FDR-sig ROIs (q<0.05): [np.int64(91)] | max|r|=0.299


## 7.4 · Two levels, two questions — and what to confirm

- **7.2 (individual, Track B):** "*do people who like the characters similarly show similar neural
  synchrony?*" — the direct test; same cohort; end-state (no run dynamics).
- **7.3 (group, Track A):** "*when a group likes a character more, does a group neural quantity move?*" —
  run-resolved but cross-cohort and correlational (coarser; 3 groups).
- Report **beside** `04b` (sentiment) — the viewer-stance counterpart, not a replacement. Never pool.

## ASK HAYOUNG!!
- **ASK HAYOUNG!!** Elevate this `like` analysis alongside the sentiment IS-RSA, and which level headlines?
- **ASK HAYOUNG!!** For 7.3, is the group-level correlational bridge (3 scramble groups) informative enough
  to report, or individual-only (7.2)?
- **ASK HAYOUNG!!** `like` similarity metric (AnnaK `−|Δ|` vs mean) and the neural collapse over runs —
  same open choices as `04c`.

## Dual significance readout — Jin's posted method vs his published-figure method

Jin's **posted** `step04.py` uses a **two-sided** bootstrap p at **FDR q<0.01**; his **published Figure 2** used a **one-sided** p at **q<0.05** (confirmed in 04a: his `.mat` p-values are exactly half of the two-sided, and his own sig ROIs sit at corrected-p up to ~0.045). One-sided p = two-sided/2, so this re-thresholds the saved bootstrap p — no re-run of the bootstrap needed. Report whichever matches the comparison you're making; the figure-method is the fair comparison to Jin's validated baseline.

In [6]:
# figure-matching readout (one-sided, q<0.05) for the individual `like` IS-RSA (7.2)
import numpy as np
from statsmodels.stats.multitest import multipletests
d=np.load("results/IS-RSA/07__like_isrsa.npy",allow_pickle=True).item()
p=np.asarray(d["p"],float)
posted=np.where(np.asarray(d["p_fdr"],float)<0.01)[0]
_,pcf,_,_=multipletests(np.minimum(p/2,1.0),alpha=0.05,method="fdr_bh")
figure=np.where(pcf<0.05)[0]
print(f"7.2 individual like  posted(2s,q<.01): {list(posted)}  | figure-match(1s,q<.05): {list(figure)}")
print("7.3 group-level already uses q<0.05 (n=3 groups -> descriptive)")

7.2 individual like  posted(2s,q<.01): [np.int64(24), np.int64(48), np.int64(60)]  | figure-match(1s,q<.05): [np.int64(24), np.int64(48), np.int64(60), np.int64(114)]
7.3 group-level already uses q<0.05 (n=3 groups -> descriptive)


## 7.5 · Noise ceiling — is the `like` effect just *better measurement*?

**The objection.** The dissociation (`like` tracks neural synchrony, sentiment does not) is a *construct* finding only if both constructs were measured with comparable precision. If `like` is simply more reliable, "brains track liking, not sentiment" collapses into "we measured liking better." No RSA effect can exceed the reliability of its own behavioural measure, so every effect must be read against its ceiling.

**Why not the standard (Nili et al., 2014) noise ceiling.** That procedure correlates *each subject's* RDM against the group-mean RDM. It does not apply here: in **inter-subject** RSA the behavioural RDM is a single subject×subject matrix, so there is no per-subject RDM to leave out. We therefore use split-half reliability, Spearman-Brown corrected, with the ceiling as sqrt(reliability).

**Design choices, and why.**
- **Split by character, not by run.** `like` is a single end-of-scan rating per subject×character — it has no repeated measurements to split temporally. Character-splitting is the only split available to *both* constructs. *Caveat:* this conflates measurement noise with genuine between-character heterogeneity; it is a conservative (downward-biased) reliability estimate for both.
- **All three 2-vs-2 character partitions, averaged** — not one arbitrary split.
- **Matched similarity metric.** `like` is scalar (AnnaK `−|Δ|`) while sentiment is a vector (cosine); those are not commensurable. So sentiment is *also* reduced to a scalar valence (pos−neg) and run through the identical AnnaK pipeline. That pair is the fair test. Vector/cosine versions are reported separately.
- **Effects compared at matched ROIs and as mean\|r\|**, not peak-across-116 (a maximum statistic is upward-biased and the bias differs by measure).
- **2000-sample bootstrap CIs** on every reliability estimate.

> [!important] Result (2026-07-18) — the dissociation is NOT a measurement artifact
> **Matched metric (scalar → AnnaK):**
> | measure | reliability (SB) | 95% CI | ceiling |
> |---|---|---|---|
> | `like` (viewer stance) | 0.343 | [0.171, 0.468] | 0.586 |
> | sentiment valence | **0.364** | [0.206, 0.480] | 0.604 |
>
> **Sentiment is measured slightly *better* than `like`**, with near-total CI overlap. The measurement-confound explanation is therefore dead — it runs the wrong way.
>
> **Ceiling-normalised effects at the same ROIs [24, 48, 60]:** `like` = 0.146 (**25% of ceiling**); sentiment = 0.019 (**4%**); embedding = 0.020 (**3%**). A ~6× difference in normalised effect at equal-or-better measurement precision.
>
> **Honest limits:** CIs are wide (n=29, 4 characters), so reliability is imprecisely estimated; and the character split cannot separate measurement noise from between-character heterogeneity. Both caveats apply *equally* to all measures, so the comparison stands even though each absolute value is uncertain.
>
> *Supersedes an earlier version of this cell that used a single arbitrary character split and cited Nili et al. incorrectly. That version gave ceilings of 0.476/0.454 and happened to favour the conclusion; the corrected estimates (0.586/0.604) reverse which measure is nominally more reliable — and strengthen the result.*


In [7]:
import numpy as np, pandas as pd, json as _json, re, sys
sys.path.insert(0,'jin_code')
from scipy.stats import spearmanr
CHAR=['jack','kate','randall','kevin']; RNG=np.random.default_rng(42); NBOOT=2000
overlap=_json.load(open('results/jinrep/03__isrsa_subject_order.json'))
con=pd.read_csv('results/step2/02__final_impression_constructs.csv')
con['pid']=con.Participant.map(lambda s: re.sub(r'\D','',str(s))); con['Character']=con.Character.str.lower().str.strip()
sc=pd.read_csv('results/scored/00__reflection_sentiment.csv')
sc['pid']=sc.Participant.map(lambda s: re.sub(r'\D','',str(s))); sc['Character']=sc.Character.str.lower().str.strip()
sc['val']=sc['Twitter_RoB_pos']-sc['Twitter_RoB_neg']
SENT=np.load('results/jinrep/03__sentiment_sim_byrun_bychar.npy',allow_pickle=True).item()
EMB =np.load('results/jinrep/03b__roberta_embed_sim_byrun_bychar.npy',allow_pickle=True).item()
def annak(vals):
    v=np.asarray(vals,float); n=len(v)
    return np.array([-abs(v[i]-v[j]) for i in range(n) for j in range(i+1,n)])
def like_by_char(g):
    subs=[re.sub(r'\D','',s) for s in overlap[str(g)]]
    return {ci: annak([(lambda x: float(x[0]) if len(x) else np.nan)(con[(con.pid==p)&(con.Character==ch)]['t_like'].values) for p in subs]) for ci,ch in enumerate(CHAR)}
def sentscalar_by_char(g):        # MATCHED METRIC: scalar valence -> AnnaK, identical to `like`
    subs=[re.sub(r'\D','',s) for s in overlap[str(g)]]
    return {ci: annak([(lambda x: float(np.nanmean(x)) if len(x) else np.nan)(sc[(sc.pid==p)&(sc.Character==ch)]['val'].values) for p in subs]) for ci,ch in enumerate(CHAR)}
mkvec=lambda D: (lambda g: {ci: np.nanmean(np.vstack([np.asarray(D[g,r],float)[ci] for r in range(1,11)]),0) for ci in range(4)})
def reliability(fn,label,nboot=NBOOT):
    per_split=[]; stacks=[]
    for half in [(0,2),(0,1),(0,3)]:                      # all three distinct 2v2 partitions
        other=tuple(sorted(set(range(4))-set(half))); A,B=[],[]
        for g in [1,2,3]:
            d=fn(g); A.append(np.nanmean(np.vstack([d[i] for i in half]),0)); B.append(np.nanmean(np.vstack([d[i] for i in other]),0))
        a,b=np.concatenate(A),np.concatenate(B); m=~(np.isnan(a)|np.isnan(b))
        r=spearmanr(a[m],b[m])[0]; per_split.append((2*r)/(1+r) if r>-1 else np.nan); stacks.append((a[m],b[m]))
    sb=float(np.nanmean(per_split)); boots=[]
    for _ in range(nboot):
        vals=[]
        for a,b in stacks:
            idx=RNG.integers(0,len(a),len(a)); rr=spearmanr(a[idx],b[idx])[0]
            vals.append((2*rr)/(1+rr) if rr>-1 else np.nan)
        boots.append(np.nanmean(vals))
    lo,hi=np.nanpercentile(boots,[2.5,97.5]); ceil=np.sqrt(max(sb,0))
    print(f'  {label:32s} SB={sb:+.3f} [{lo:+.3f},{hi:+.3f}]  ceiling={ceil:.3f}  per-split={[round(x,3) for x in per_split]}')
    return sb,ceil,(np.sqrt(max(lo,0)),np.sqrt(max(hi,0)))
print('RELIABILITY (all 3 character splits averaged, 2000-boot CI)')
print('-- matched metric (scalar -> AnnaK): the fair like-vs-sentiment test --')
rel_like=reliability(like_by_char,'LIKE (viewer stance)')
rel_sent=reliability(sentscalar_by_char,'sentiment valence (scalar)')
print('-- native vector metric (cosine) --')
rel_sentv=reliability(mkvec(SENT),'sentiment 3-D (cosine)')
rel_emb  =reliability(mkvec(EMB),'RoBERTa 768-D (cosine)')
print('\nCEILING-NORMALISED EFFECTS (mean|r|, not peak; at matched ROIs)')
LIKE_ROIS=[24,48,60]; summary={}
for lab,path,rel in [('LIKE','results/IS-RSA/07__like_isrsa.npy',rel_like),
                     ('sentiment','results/IS-RSA/04b__sentiment_isrsa_after.npy',rel_sentv),
                     ('embedding','results/IS-RSA/04b.1__sentiment_isrsa_after.npy',rel_emb)]:
    d=np.load(path,allow_pickle=True).item(); mr=np.abs(np.asarray(d['mean_r'],float)); ceil=rel[1]
    a116=float(np.nanmean(mr)); aroi=float(np.nanmean(mr[LIKE_ROIS]))
    summary[lab]={'reliability_SB':rel[0],'ceiling':ceil,'ceiling_CI':rel[2],'mean_r_all116':a116,'mean_r_likeROIs':aroi,
                  'pct_ceiling_all116':100*a116/ceil,'pct_ceiling_likeROIs':100*aroi/ceil}
    print(f'  {lab:10s} ceiling={ceil:.3f} | all116={a116:.3f} ({100*a116/ceil:.0f}%) | @[24,48,60]={aroi:.3f} ({100*aroi/ceil:.0f}%)')
np.save('results/step5/07__noise_ceilings.npy',summary,allow_pickle=True)
print('\nsaved -> results/step5/07__noise_ceilings.npy')


RELIABILITY (all 3 character splits averaged, 2000-boot CI)
-- matched metric (scalar -> AnnaK): the fair like-vs-sentiment test --


/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_16073/1922845240.py:28: RuntimeWarning: Mean of empty slice
  d=fn(g); A.append(np.nanmean(np.vstack([d[i] for i in half]),0)); B.append(np.nanmean(np.vstack([d[i] for i in other]),0))


  LIKE (viewer stance)             SB=+0.343 [+0.171,+0.468]  ceiling=0.586  per-split=[np.float64(0.226), np.float64(0.411), np.float64(0.392)]
  sentiment valence (scalar)       SB=+0.364 [+0.206,+0.480]  ceiling=0.604  per-split=[np.float64(0.399), np.float64(0.313), np.float64(0.381)]
-- native vector metric (cosine) --
  sentiment 3-D (cosine)           SB=+0.256 [+0.069,+0.384]  ceiling=0.506  per-split=[np.float64(0.206), np.float64(0.282), np.float64(0.28)]
  RoBERTa 768-D (cosine)           SB=+0.334 [+0.164,+0.448]  ceiling=0.578  per-split=[np.float64(0.375), np.float64(0.276), np.float64(0.351)]

CEILING-NORMALISED EFFECTS (mean|r|, not peak; at matched ROIs)
  LIKE       ceiling=0.586 | all116=0.046 (8%) | @[24,48,60]=0.146 (25%)
  sentiment  ceiling=0.506 | all116=0.015 (3%) | @[24,48,60]=0.019 (4%)
  embedding  ceiling=0.578 | all116=0.016 (3%) | @[24,48,60]=0.020 (3%)

saved -> results/step5/07__noise_ceilings.npy
